# Data agent

Ask questions about your own business data. The agent finds relevant warehouse tables, checks metric definitions and past analyses, runs read-only SQL, and remembers useful corrections for future investigations.

```mermaid
sequenceDiagram
    participant Person
    participant App as Your application
    participant Agent as Agents API
    participant Warehouse as Read-only warehouse
    participant Memory as Analyst memory

    Person->>App: Why did paid conversions drop last week?
    App->>Agent: Start or resume the user's session
    Agent->>Warehouse: Discover tables and inspect their schemas
    Agent->>App: Check definitions, company knowledge, and past queries
    Agent->>Memory: Find saved analyst corrections
    Agent->>Warehouse: Execute and verify read-only SQL
    Agent-->>Person: Explain the findings, sources, and assumptions
    Person->>Agent: Remember to exclude manually provisioned accounts.
    Agent->>Memory: Save the correction for future analyses
```


## Agents API capabilities

Function tools, Tool search, Programmatic tool calling, Persistent sessions, Streaming.

### Find the right data

The agent discovers available tables and inspects real schemas instead of assuming where business data lives.

### Use business context

Metric definitions, trusted previous queries, and company documents explain what the numbers actually mean.

### Verify every answer

Your application owns the warehouse connection, enforces read-only access, and exposes every executed query.

### Continue the investigation

A persistent session keeps the original question, findings, and follow-ups connected.

### Remember what the team learns

Explicit corrections can be saved as personal or shared team memory and reused in future analyses.


## Application flow

1. Business question.
2. Warehouse catalog.
3. Business context.
4. Agent session.
5. Read-only SQL.
6. Verified answer.
7. Saved memory.


## What you need

- Python 3.14+ and `uv`.
- An OpenAI API key.
- Read-only access to a PostgreSQL-compatible warehouse.

No execution sandbox is required. The agent uses an Agents API session without an environment, and your application controls every warehouse query.


## Connect your warehouse

From the repository root:

```bash
cp examples/agents_api/apps/data_analyst/.env.example examples/agents_api/apps/data_analyst/.env
```

Add your OpenAI API key and read-only `WAREHOUSE_URL` to `examples/agents_api/apps/data_analyst/.env`. The application loads this file automatically.

Optionally, uncomment `DATA_AGENT_CONTEXT` in `.env` and adapt the example context file with your team's table descriptions, metric definitions, previous queries, and company documents. Saved analyst corrections are stored in `examples/agents_api/apps/data_analyst/memories.json`.


## Run the data agent

```bash
uv run examples/agents_api/apps/data_analyst/main.py
```

Open [http://127.0.0.1:8000](http://127.0.0.1:8000) and ask:

```text
Why did paid conversions drop last week?
```

Then continue the same investigation:

```text
Only include enterprise customers.
```

Or ask a single question from the terminal:

```bash
uv run examples/agents_api/apps/data_analyst/main.py --prompt \
  "Why did paid conversions drop last week?"
```


## How it works

The agent has four focused tools:

- `search_tables` finds relevant tables and reads their actual columns.
- `search_context` retrieves metric definitions, prior queries, documents, and saved corrections.
- `query_warehouse` executes one read-only query and returns at most 100 rows.
- `save_memory` stores an explicitly requested correction for later.

The `save_memory` tool is discovered on demand with `tool_search`. Programmatic tool calling lets the agent combine metadata, business context, saved corrections, and multiple verified queries before responding.

Use a read-only warehouse role. The application also starts PostgreSQL connections in read-only mode, applies a query timeout, and shows every executed query. The included browser interface represents one local analyst. In a multi-user application, derive analyst identity from your authentication layer, not a request body or query parameter.


## Run this notebook

Use a Jupyter Python kernel (Python 3.11 or later) on macOS or Linux in a local clone of the [Cookbook repository](https://github.com/openai/openai-cookbook). The application itself uses Python 3.14; `uv run` installs the dependencies declared in `main.py` and selects that interpreter.

The terminal commands above run the checked-in application. The notebook instead builds a separate copy inside an ignored `tmp_` workspace under your Cookbook checkout. Each `%%writefile` cell contains actual application source. Run these cells in order: the first cell for a module creates its file, and later cells append to it. Python definitions are executed by the application when you launch it.

The setup cell copies only the listed supporting fixtures, manifests, and policy files. It creates a fresh `.env` from the example template without copying your existing credentials. Configure the printed `.env` path before the optional launch step. Rerunning setup creates a new workspace; keep the previous workspace if you need its reports or memory.

Default execution builds and checks the files locally. Docker builds and live API calls require the explicit flags in the launch section.


In [ ]:
from pathlib import Path
import shutil
import tempfile

if globals().get("application_process") is not None and application_process.poll() is None:
    raise RuntimeError("Stop the running application before creating a new workspace.")
application_process = None

# Start Jupyter anywhere inside the Cookbook checkout.
working_directory = Path.cwd().resolve()
cookbook_root = next(
    (path for path in [working_directory, *working_directory.parents]
     if (path / "examples/agents_api/apps/data_analyst/main.py").is_file()),
    None,
)
if cookbook_root is None:
    raise FileNotFoundError("Clone openai/openai-cookbook and start Jupyter inside it.")

notebook_root = Path(tempfile.mkdtemp(prefix="tmp_agents_data_analyst_", dir=cookbook_root))
application_dir = notebook_root / "examples/agents_api/apps/data_analyst"
application_dir.mkdir(parents=True)
for package in [notebook_root / "examples", notebook_root / "examples/agents_api",
                notebook_root / "examples/agents_api/apps", application_dir]:
    (package / "__init__.py").touch()

support_paths = [
    ".env.example",
    "context.example.json",
    "index.html"
]
source_dir = cookbook_root / "examples/agents_api/apps/data_analyst"
for relative_path in support_paths:
    destination = application_dir / relative_path
    destination.parent.mkdir(parents=True, exist_ok=True)
    shutil.copy2(source_dir / relative_path, destination)
shutil.copy2(application_dir / ".env.example", application_dir / ".env")
shutil.copytree(
    cookbook_root / "examples/agents_api/sandboxes/application_managed/docker",
    notebook_root / "examples/agents_api/sandboxes/application_managed/docker",
)
print(f"Application workspace: {application_dir}")
print(f"Configure credentials in: {application_dir / '.env'}")


## Implementation walkthrough

This walkthrough covers warehouse discovery, read-only SQL, business context, persistent sessions, and explicit analyst memory.

Follow the setup instructions above, then build the application with the code cells below.


### 1. Connect your warehouse

Clone the repository, copy the example's environment template, and configure a read-only warehouse account in its .env file.

Customize the optional context file to match your warehouse. The application does not download or create a fixed dataset.


### 2. Discover the right tables

Give the agent one discovery tool that finds relevant tables and returns their actual columns, along with any ownership and freshness metadata supplied in your context file.

![A data agent discovering warehouse tables, schemas, metrics, and business context.](../../../../images/agents_api/agents-api-data-agent-catalog.webp)


### 3. Add the context behind the numbers

A column name rarely tells the full story. Give the agent metric definitions, trusted filters, and company notes that explain how your team interprets the data.


### 4. Reuse trusted previous analyses

Previous reviewed queries help the agent find established joins and familiar calculation patterns instead of reinventing a business metric.

Match these example tables and columns to your warehouse. Add an explicit signup date range for weekly comparisons.


### 5. Keep warehouse access read-only

The agent never receives database credentials. Your application runs each approved query using a read-only warehouse account, blocks write statements, applies a timeout, and returns at most 100 rows.

Use a read-only warehouse role as the security boundary. The application also configures the PostgreSQL connection as read-only.

![An agent validating read-only SQL against warehouse data and presenting an evidence-backed trend.](../../../../images/agents_api/agents-api-data-agent-analysis.webp)


### 6. Register four focused tools

Keep the application-owned tool surface small. Discovery returns real schemas, context includes prior work and memories, and memory writing is loaded only when needed.


### 7. Start an investigation

Create an Agents API session without a sandbox. The model uses your application-owned tools to discover context, check the data, and return a verified answer.


### 8. Continue the same conversation

Retrieve the original session when the user asks a follow-up. The agent keeps the investigation context and can narrow the answer without starting over.


### 9. Remember useful analyst corrections

When someone explicitly asks the agent to remember a rule, save it as personal or shared team memory. Later investigations can retrieve that correction before querying the warehouse.

![An analyst correction saved as persistent memory and reused during a later investigation.](../../../../images/agents_api/agents-api-data-agent-memory.webp)


### 10. Run the data agent

Start the browser interface to investigate interactively, or run one question directly from the terminal.


### 11. Close completed investigations

Delete the session and close the warehouse connection when the investigation is complete or the application shuts down.


## Build the application

The following cells include every application module. Run all cells for each file before launching. The inline dependency declaration in `main.py` installs the current OpenAI SDK and the application libraries through `uv`.


### memory.py

Persist explicit analyst corrections and filter personal or team memories for the current user.


In [ ]:
%%writefile "{application_dir}/memory.py"
"""Search and persist personal or team analyst corrections."""

from __future__ import annotations

import json
import re
from datetime import UTC, datetime
from pathlib import Path
from typing import Any, cast
from uuid import uuid4


def relevant(record: dict[str, Any], query: str) -> bool:
    terms = {word for word in re.findall(r"[a-z0-9_]+", query.lower()) if len(word) > 2}
    if not terms:
        return True
    text = json.dumps(record, default=str).lower()
    return any(term in text for term in terms)


class MemoryStore:
    """Keep explicitly saved analyst corrections in a local JSON file."""

    def __init__(self, path: Path) -> None:
        self.path = path

    def list(self, *, user_id: str = "analyst") -> list[dict[str, Any]]:
        if not self.path.exists():
            return []
        memories = json.loads(self.path.read_text())
        if not isinstance(memories, list):
            return []
        return [
            cast(dict[str, Any], memory)
            for memory in memories
            if isinstance(memory, dict)
            and (memory.get("scope") == "team" or memory.get("user_id") == user_id)
        ]

    def search(
        self, arguments: dict[str, Any], *, user_id: str = "analyst"
    ) -> dict[str, Any]:
        query = str(arguments.get("query", ""))
        return {
            "memories": [
                memory
                for memory in self.list(user_id=user_id)
                if relevant(memory, query)
            ][:10]
        }

    def save(
        self, arguments: dict[str, Any], *, user_id: str = "analyst"
    ) -> dict[str, Any]:
        note = str(arguments.get("note", "")).strip()
        scope = str(arguments.get("scope", "personal"))
        if not note:
            raise ValueError("A saved memory must contain a note.")
        if scope not in {"personal", "team"}:
            raise ValueError("A memory scope must be personal or team.")

        memories = (
            cast(list[dict[str, Any]], json.loads(self.path.read_text()))
            if self.path.exists()
            else []
        )
        memory: dict[str, Any] = {
            "id": uuid4().hex,
            "note": note,
            "scope": scope,
            "user_id": user_id,
            "created_at": datetime.now(UTC).isoformat(),
        }
        memories.append(memory)
        self.path.parent.mkdir(parents=True, exist_ok=True)
        self.path.write_text(json.dumps(memories, indent=2) + "\n")
        return {"status": "saved", "memory": memory}

    def delete(self, memory_id: str, *, user_id: str = "analyst") -> dict[str, Any]:
        memory = next(
            (
                memory
                for memory in self.list(user_id=user_id)
                if memory.get("id") == memory_id
            ),
            None,
        )
        if memory is None:
            raise ValueError("Memory not found.")

        memories = cast(list[dict[str, Any]], json.loads(self.path.read_text()))
        memories = [item for item in memories if item.get("id") != memory_id]
        self.path.write_text(json.dumps(memories, indent=2) + "\n")
        return {"status": "deleted", "memory": memory}


### warehouse.py

Discover tables, load business context, and execute bounded queries through a read-only database connection.


In [ ]:
%%writefile "{application_dir}/warehouse.py"
"""Inspect warehouse schemas and run read-only queries."""

from __future__ import annotations

import importlib
import json
import os
import re
import sqlite3
from pathlib import Path
from typing import Any, cast
from urllib.parse import unquote, urlparse

from .memory import MemoryStore, relevant

EXAMPLE_DIR = Path(__file__).resolve().parent
MAX_ROWS = 100


class Warehouse:
    """Expose an existing warehouse through a read-only, application-owned connection."""

    def __init__(
        self,
        url: str | None = None,
        *,
        context: dict[str, Any] | None = None,
        context_path: Path | None = None,
        memory_path: Path | None = None,
    ) -> None:
        self.url = url or os.environ.get("WAREHOUSE_URL")
        if not self.url:
            raise ValueError("Set WAREHOUSE_URL to a read-only PostgreSQL connection.")

        configured_context = os.environ.get("DATA_AGENT_CONTEXT")
        if context is not None:
            self.context = context
        elif context_path is not None or configured_context:
            path = context_path or Path(str(configured_context))
            self.context = cast(dict[str, Any], json.loads(path.read_text()))
        else:
            self.context = {}

        self.memory = MemoryStore(memory_path or EXAMPLE_DIR / "memories.json")

        parsed = urlparse(self.url)
        self.connection: Any
        if parsed.scheme in {"postgres", "postgresql"}:
            self.engine = "PostgreSQL"
            try:
                psycopg = importlib.import_module("psycopg")
                rows = importlib.import_module("psycopg.rows")
            except ModuleNotFoundError as error:
                raise RuntimeError(
                    "PostgreSQL requires psycopg. Start the example with "
                    "`uv run examples/agents_api/apps/data_analyst/main.py`."
                ) from error
            self.connection = psycopg.connect(
                self.url,
                autocommit=True,
                row_factory=rows.dict_row,
                options="-c default_transaction_read_only=on -c statement_timeout=30000",
            )
        elif parsed.scheme == "sqlite":
            self.engine = "SQLite"
            path = Path(unquote(parsed.path)).resolve()
            self.connection = sqlite3.connect(
                f"file:{path}?mode=ro", uri=True, check_same_thread=False
            )
            self.connection.row_factory = sqlite3.Row
        else:
            raise ValueError("WAREHOUSE_URL must use postgresql:// or sqlite:///.")

    def records(self, section: str) -> list[dict[str, Any]]:
        records = self.context.get(section, [])
        if not isinstance(records, list):
            return []
        return [
            cast(dict[str, Any], record)
            for record in records
            if isinstance(record, dict)
        ]

    def table_names(self) -> list[str]:
        cursor = self.connection.cursor()
        if self.engine == "PostgreSQL":
            cursor.execute(
                "SELECT table_schema, table_name FROM information_schema.tables "
                "WHERE table_schema NOT IN ('information_schema', 'pg_catalog') "
                "AND table_type IN ('BASE TABLE', 'VIEW') "
                "ORDER BY table_schema, table_name"
            )
            return [
                f"{row['table_schema']}.{row['table_name']}"
                for row in cursor.fetchall()
            ]

        cursor.execute(
            "SELECT name FROM sqlite_master "
            "WHERE type IN ('table', 'view') AND name NOT LIKE 'sqlite_%' ORDER BY name"
        )
        return [str(row["name"]) for row in cursor.fetchall()]



Continue `warehouse.py`: `Warehouse.table_context`, `Warehouse.search_tables`, `Warehouse.inspect_table`, `Warehouse.search_query_history`, `Warehouse.search_company_knowledge`, `Warehouse.search_context`, `Warehouse.query`. This cell appends to the same file.


In [ ]:
%%writefile -a "{application_dir}/warehouse.py"
    def table_context(self, name: str) -> dict[str, Any]:
        records = self.records("tables")
        for record in records:
            if record.get("name") == name:
                return record

        table_name = name.rsplit(".", maxsplit=1)[-1]
        return next(
            (
                record
                for record in records
                if str(record.get("name", "")).rsplit(".", maxsplit=1)[-1] == table_name
            ),
            {"name": name},
        )

    def search_tables(self, arguments: dict[str, Any]) -> dict[str, Any]:
        query = str(arguments.get("query", ""))
        names = [
            name
            for name in self.table_names()
            if relevant(self.table_context(name), query)
        ][:20]
        return {"tables": [self.inspect_table({"table": name}) for name in names]}

    def inspect_table(self, arguments: dict[str, Any]) -> dict[str, Any]:
        name = str(arguments.get("table", ""))
        if name not in self.table_names():
            raise ValueError(f"Unknown warehouse table: {name}")

        cursor = self.connection.cursor()
        if self.engine == "PostgreSQL":
            schema, table = name.split(".", maxsplit=1)
            cursor.execute(
                "SELECT column_name, data_type, is_nullable "
                "FROM information_schema.columns "
                "WHERE table_schema = %s AND table_name = %s ORDER BY ordinal_position",
                (schema, table),
            )
            columns: list[dict[str, Any]] = [
                {
                    "name": str(row["column_name"]),
                    "type": str(row["data_type"]),
                    "nullable": row["is_nullable"] == "YES",
                }
                for row in cursor.fetchall()
            ]
        else:
            escaped = name.replace('"', '""')
            cursor.execute(f'PRAGMA table_info("{escaped}")')
            columns = [
                {
                    "name": str(row["name"]),
                    "type": str(row["type"]),
                    "nullable": not bool(row["notnull"]),
                }
                for row in cursor.fetchall()
            ]

        return {**self.table_context(name), "name": name, "columns": columns}

    def search_query_history(self, arguments: dict[str, Any]) -> dict[str, Any]:
        query = str(arguments.get("query", ""))
        return {
            "queries": [
                record
                for record in self.records("query_history")
                if relevant(record, query)
            ][:10]
        }

    def search_company_knowledge(self, arguments: dict[str, Any]) -> dict[str, Any]:
        query = str(arguments.get("query", ""))
        return {
            "metrics": [
                record for record in self.records("metrics") if relevant(record, query)
            ][:10],
            "documents": [
                record
                for record in self.records("documents")
                if relevant(record, query)
            ][:10],
        }

    def search_context(
        self, arguments: dict[str, Any], *, user_id: str = "analyst"
    ) -> dict[str, Any]:
        return {
            **self.search_company_knowledge(arguments),
            **self.search_query_history(arguments),
            **self.memory.search(arguments, user_id=user_id),
        }

    def query(self, arguments: dict[str, Any]) -> dict[str, Any]:
        sql = str(arguments.get("sql", "")).strip().rstrip(";")
        if not re.match(r"^(SELECT|WITH)\b", sql, re.IGNORECASE) or ";" in sql:
            raise ValueError("Only one read-only SELECT query is allowed.")

        cursor = self.connection.cursor()
        cursor.execute(sql)
        rows = [dict(row) for row in cursor.fetchmany(MAX_ROWS)]
        serialized = cast(
            list[dict[str, Any]], json.loads(json.dumps(rows, default=str))
        )
        return {"sql": sql, "rows": serialized, "row_count": len(serialized)}



Continue `warehouse.py`: `Warehouse.summary`, `Warehouse.close`. This cell appends to the same file.


In [ ]:
%%writefile -a "{application_dir}/warehouse.py"
    def summary(self, *, user_id: str = "analyst") -> dict[str, Any]:
        return {
            "engine": self.engine,
            "tables": len(self.table_names()),
            "metrics": len(self.records("metrics")),
            "documents": len(self.records("documents")),
            "memories": len(self.memory.list(user_id=user_id)),
        }

    def close(self) -> None:
        self.connection.close()


### agent.py

Register warehouse and memory tools, stream verified answers, and retain a session for follow-up questions.


In [ ]:
%%writefile "{application_dir}/agent.py"
"""Investigate warehouse questions with persistent Agents API sessions."""

from __future__ import annotations

import asyncio
import json
import os
import sys
from collections.abc import Callable, Mapping
from typing import Any
from uuid import uuid4

from openai import AsyncOpenAI
from openai.types.beta import AgentToolParam
from openai.types.beta.agent_tool_param import AgentToolConfigParamFunction
from openai.types.beta.agents.session_create_params import Agent

from .warehouse import Warehouse

INSTRUCTIONS = """\
You are a careful business data analyst.
Find relevant warehouse tables and inspect their schemas.
Check business definitions, trusted prior queries, and saved analyst corrections.
Execute only read-only queries.
Explain your verified findings, sources, assumptions, and SQL.
Save a memory only when the user explicitly asks you to remember a correction.
"""


def define_tool(
    name: str,
    description: str,
    properties: Mapping[str, dict[str, Any]],
    *,
    defer_loading: bool = False,
) -> AgentToolConfigParamFunction:
    tool: AgentToolConfigParamFunction = {
        "type": "function",
        "name": name,
        "description": description,
        "parameters": {
            "type": "object",
            "properties": dict(properties),
            "required": list(properties),
            "additionalProperties": False,
        },
    }
    if defer_loading:
        tool["defer_loading"] = True
    return tool


TOOLS: list[AgentToolParam] = [
    define_tool(
        "search_tables",
        "Find relevant warehouse tables and inspect their actual columns and business context.",
        {"query": {"type": "string"}},
    ),
    define_tool(
        "search_context",
        "Find business definitions, reviewed queries, company documents, and analyst memories.",
        {"query": {"type": "string"}},
    ),
    define_tool(
        "query_warehouse",
        "Execute one read-only SQL query and return at most 100 rows.",
        {"sql": {"type": "string"}},
    ),
    define_tool(
        "save_memory",
        "Save a useful correction only when the user explicitly asks you to remember it.",
        {
            "note": {"type": "string"},
            "scope": {"type": "string", "enum": ["personal", "team"]},
        },
        defer_loading=True,
    ),
    {"type": "tool_search"},
    {"type": "programmatic_tool_calling", "enabled": True},
]


class DataAnalyst:
    """Keep one Agents API session for each warehouse investigation."""

    def __init__(self, client: AsyncOpenAI, warehouse: Warehouse) -> None:
        self.client = client
        self.warehouse = warehouse
        self.model = os.environ.get("OPENAI_MODEL", "gpt-5.6-luna")
        self.sessions: dict[str, str] = {}
        self.owners: dict[str, str] = {}

    async def answer(
        self,
        question: str,
        conversation_id: str | None = None,
        *,
        user_id: str = "analyst",
    ) -> dict[str, Any]:
        conversation_id = conversation_id or uuid4().hex
        owner = self.owners.get(conversation_id)
        if owner is not None and owner != user_id:
            raise PermissionError(
                "Start a new conversation to use your own analyst context."
            )

        queries: list[str] = []

        def execute(arguments: dict[str, Any]) -> dict[str, Any]:
            result = self.warehouse.query(arguments)
            queries.append(str(result["sql"]))
            return result

        handlers: dict[str, Callable[[dict[str, Any]], dict[str, Any]]] = {
            "search_tables": self.warehouse.search_tables,
            "search_context": lambda arguments: self.warehouse.search_context(
                arguments, user_id=user_id
            ),
            "query_warehouse": execute,
            "save_memory": lambda arguments: self.warehouse.memory.save(
                arguments, user_id=user_id
            ),
        }

        first_turn = conversation_id not in self.sessions
        session_id = self.sessions.get(conversation_id)
        if session_id is not None:
            session = await self.client.beta.agents.sessions.retrieve(session_id)
            events = self.client.beta.agents.sessions.stream(
                session.id, input=question, tool_handlers=handlers
            )
        else:
            agent: Agent = {
                "model": self.model,
                "instructions": INSTRUCTIONS,
                "reasoning": {"effort": "high"},
                "tools": TOOLS,
            }
            # Conversation-only sessions need input at creation, not in a later call.
            events = await self.client.beta.agents.sessions.create(
                agent=agent,
                environment={"type": "none"},
                input=question,
                stream=True,
            )

        parts: list[str] = []
        completed = False
        handled_calls: set[tuple[str, str]] = set()
        async with events:
            async for event in events:
                if event.type == "agent.session.created":
                    session_id = event.session.id
                    self.sessions[conversation_id] = session_id
                    self.owners[conversation_id] = user_id
                elif first_turn and event.type == "agent.session.requires_action":
                    # Creation streams expose pending calls; follow-ups use SDK handlers.
                    for action in event.session.required_actions:
                        if action.type != "function_call":
                            continue
                        call = (action.turn_id, action.call_id)
                        if call in handled_calls:
                            continue
                        try:
                            arguments = action.arguments
                            if isinstance(arguments, str):
                                arguments = json.loads(arguments)
                            if not isinstance(arguments, dict):
                                raise ValueError("Function arguments must be an object")
                            output = json.dumps(handlers[action.name](arguments))
                            success, error = True, None
                        except Exception:
                            output, success, error = None, False, "Tool handler failed."
                        await self.client.beta.agents.sessions.events.create(
                            event.session.id,
                            events=[
                                {
                                    "type": "agent.session.input.tool_result",
                                    "turn_id": action.turn_id,
                                    "call_id": action.call_id,
                                    "success": success,
                                    "output": output,
                                    "error": error,
                                }
                            ],
                            idempotency_key=str(uuid4()),
                        )
                        handled_calls.add(call)
                elif event.type == "agent.session.turn.output_text.delta":
                    parts.append(event.delta)
                elif event.type == "agent.session.turn.output_text.done" and not parts:
                    parts.append(event.text)
                elif event.type in {
                    "agent.session.failed",
                    "agent.session.turn.failed",
                    "error",
                }:
                    raise RuntimeError(f"Data investigation failed: {event.to_dict()}")
                elif event.type == "agent.session.turn.cancelled":
                    raise RuntimeError("Data investigation was cancelled.")
                elif (
                    event.type == "agent.session.turn.completed"
                    and event.turn.subagent_id is None
                ):
                    completed = True
        if not completed or session_id is None:
            raise RuntimeError("Stream ended without a completed investigation.")
        answer = "".join(parts)

        return {
            "answer": answer,
            "conversation_id": conversation_id,
            "session_id": session_id,
            "sql": queries[-1] if queries else None,
            "queries": queries,
            "memories": self.warehouse.memory.list(user_id=user_id),
        }



Continue `agent.py`: `DataAnalyst.close`. This cell appends to the same file.


In [ ]:
%%writefile -a "{application_dir}/agent.py"
    async def close(self) -> None:
        original_error = sys.exception()
        session_ids = list(set(self.sessions.values()))
        try:
            results = await asyncio.gather(
                *(self.client.beta.agents.sessions.delete(sid) for sid in session_ids),
                return_exceptions=True,
            )
            for session_id, result in zip(session_ids, results):
                if isinstance(result, BaseException):
                    if original_error is not None:
                        original_error.add_note(
                            f"Could not delete session {session_id}: {result}"
                        )
                        continue
                    raise result
        finally:
            self.sessions.clear()
            self.owners.clear()
            self.warehouse.close()


### main.py

Expose the analyst through FastAPI or a single-command prompt, and close sessions when the application exits.


In [ ]:
%%writefile "{application_dir}/main.py"
# /// script
# requires-python = ">=3.14"
# dependencies = [
#     "openai>=3.13.0",
#     "fastapi",
#     "psycopg[binary]",
#     "python-dotenv",
#     "uvicorn",
# ]
# ///

"""Serve the data analyst UI or run a single question."""

from __future__ import annotations

import argparse
import asyncio
import sys
from collections.abc import AsyncIterator
from contextlib import asynccontextmanager
from pathlib import Path
from typing import Any, cast

from dotenv import load_dotenv
from fastapi import FastAPI, HTTPException
from fastapi.responses import HTMLResponse
from openai import AsyncOpenAI
from pydantic import BaseModel

# Support direct execution from any working directory.
if __package__ in {None, ""}:
    sys.path.insert(0, str(Path(__file__).resolve().parents[4]))


from examples.agents_api.apps.data_analyst.agent import DataAnalyst
from examples.agents_api.apps.data_analyst.warehouse import Warehouse

EXAMPLE_DIR = Path(__file__).resolve().parent


@asynccontextmanager
async def lifespan(app: FastAPI) -> AsyncIterator[None]:
    async with AsyncOpenAI() as client:
        app.state.analyst = DataAnalyst(client, Warehouse())
        try:
            yield
        finally:
            await app.state.analyst.close()


app = FastAPI(title="Data agent", lifespan=lifespan)


class Question(BaseModel):
    question: str
    conversation_id: str | None = None


class Memory(BaseModel):
    note: str
    scope: str = "personal"


@app.get("/", response_class=HTMLResponse)
async def home() -> str:
    return (EXAMPLE_DIR / "index.html").read_text()


@app.get("/api/warehouse")
async def warehouse_summary() -> dict[str, Any]:
    analyst: DataAnalyst = app.state.analyst
    return analyst.warehouse.summary()


@app.get("/api/memories")
async def list_memories() -> dict[str, Any]:
    analyst: DataAnalyst = app.state.analyst
    return {"memories": analyst.warehouse.memory.list()}


@app.post("/api/memories")
async def create_memory(memory: Memory) -> dict[str, Any]:
    analyst: DataAnalyst = app.state.analyst
    try:
        return analyst.warehouse.memory.save(
            {"note": memory.note, "scope": memory.scope}
        )
    except ValueError as error:
        raise HTTPException(status_code=400, detail=str(error)) from error


@app.delete("/api/memories/{memory_id}")
async def delete_memory(memory_id: str) -> dict[str, Any]:
    analyst: DataAnalyst = app.state.analyst
    try:
        analyst.warehouse.memory.delete(memory_id)
    except ValueError as error:
        raise HTTPException(status_code=404, detail=str(error)) from error
    return {"memories": analyst.warehouse.memory.list()}




Continue `main.py`: `ask`, `run_prompt`, `main`, `Application entry point`. This cell appends to the same file.


In [ ]:
%%writefile -a "{application_dir}/main.py"
@app.post("/api/ask")
async def ask(question: Question) -> dict[str, Any]:
    analyst: DataAnalyst = app.state.analyst
    try:
        return await analyst.answer(question.question, question.conversation_id)
    except (PermissionError, ValueError) as error:
        raise HTTPException(status_code=400, detail=str(error)) from error


async def run_prompt(prompt: str) -> None:
    async with AsyncOpenAI() as client:
        analyst = DataAnalyst(client, Warehouse())
        try:
            result = await analyst.answer(prompt)
            print(result["answer"])
            queries = cast(list[str], result["queries"])
            if queries:
                print("\nVerified SQL:")
                for query in queries:
                    print(query)
        finally:
            await analyst.close()


def main() -> None:
    load_dotenv(EXAMPLE_DIR / ".env")
    parser = argparse.ArgumentParser(
        description="Ask questions about a read-only data warehouse."
    )
    parser.add_argument(
        "--prompt", metavar="QUESTION", help="Run one warehouse investigation."
    )
    args = parser.parse_args()

    if args.prompt:
        asyncio.run(run_prompt(args.prompt))
        return

    import uvicorn

    uvicorn.run(app, host="127.0.0.1", port=8000)


if __name__ == "__main__":
    main()


## Check the generated files

Compile all generated modules without importing them or contacting external services. This catches syntax errors before you launch the application.


In [ ]:
import py_compile

modules = ["memory.py", "warehouse.py", "agent.py", "main.py"]
for filename in modules:
    py_compile.compile(str(application_dir / filename), doraise=True)
print(f"Compiled {len(modules)} application modules.")


## Launch the application (optional)

Edit the generated `.env` file with the credentials listed above. Install `uv` and, for sandbox applications, start Docker. The following cells are disabled by default. Enabling them may incur API usage and connect to the configured services.

Use the generated workspace for every path below. For a hosted deployment, package the generated application files and supply credentials through your deployment's secret configuration.


The launch arguments ask one question against your configured warehouse. Set `application_arguments = []` to run the persistent data analyst web UI instead. The process writes to `application.log` in the generated workspace. Inspect that file for errors and progress.


In [ ]:
import os
import subprocess

RUN_APPLICATION = False
application_arguments = ["--prompt", "Which tables should I use to investigate weekly revenue?"]
if RUN_APPLICATION:
    if application_process is not None and application_process.poll() is None:
        raise RuntimeError("Stop the previous application before launching again.")
    with (application_dir / "application.log").open("w") as application_log:
        application_process = subprocess.Popen(
            ["uv", "run", str(application_dir / "main.py"), *application_arguments],
            cwd=notebook_root,
            env={key: value for key, value in os.environ.items() if key != "VIRTUAL_ENV"},
            start_new_session=True,
            stdout=application_log,
            stderr=subprocess.STDOUT,
        )
    print(f"Process started: {application_process.pid}")
    print(f"Progress log: {application_dir / 'application.log'}")


### Stop a running application

Set `STOP_APPLICATION = True` after you finish. Interrupt the application process group so the application's shutdown handlers can close sessions and remove containers. Batch commands normally exit on their own. A timeout means shutdown is still in progress; inspect the log before taking further action.


In [ ]:
import os
import signal

STOP_APPLICATION = False
if STOP_APPLICATION and application_process is not None:
    if application_process.poll() is None:
        os.killpg(os.getpgid(application_process.pid), signal.SIGINT)
        application_process.wait(timeout=30)
    print(f"Application exited with status {application_process.returncode}.")


The generated workspace remains available for reports and memory. Remove it manually after stopping the application and saving any files you need. Do not rerun the launch cell while the previous process is running.


## Example result

An illustrative answer is shown below. Your results come from your warehouse, with assumptions and executed SQL available for review.

The following illustrates a possible result; model-generated findings depend on the inputs and connected sources.

```text
Paid conversion fell from 12.8% to 10.1% last week.

Primary driver:
Mobile checkout conversion declined after the August 18 release.

Sources:
analytics.signups
analytics.subscriptions
Checkout tracking migration notes

Assumptions:
Internal and test accounts excluded.
Incomplete current-day data excluded.

SQL:
SELECT signup_week, channel, COUNT(*) AS signups, ...
```


## Next steps

- Connect your approved warehouse, semantic layer, or analytics catalog.
- Add the metric definitions, trusted queries, and business documents your team already uses.
- Derive analyst identity from authentication, then apply warehouse and tenant permissions.
- Store shared analyst memories in your existing application database.


## Related documentation

- [Inside OpenAI's in-house data agent](https://openai.com/index/inside-our-in-house-data-agent/): How OpenAI combines warehouse metadata, business context, analyst memory, and transparent data investigations.


## Files

- [main.py](https://github.com/openai/openai-cookbook/blob/main/examples/agents_api/apps/data_analyst/main.py): Web routes, UI, and the command-line entrypoint.
- [agent.py](https://github.com/openai/openai-cookbook/blob/main/examples/agents_api/apps/data_analyst/agent.py): Agent configuration, tools, and conversation flow.
- [warehouse.py](https://github.com/openai/openai-cookbook/blob/main/examples/agents_api/apps/data_analyst/warehouse.py): Read-only queries, schemas, and business context.
- [memory.py](https://github.com/openai/openai-cookbook/blob/main/examples/agents_api/apps/data_analyst/memory.py): Saved personal and team corrections.
